In [ ]:
project_path = "/home/jupyter"
import os
import sys

sys.path.append(project_path)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import plotly.express as px
from datetime import datetime
from google.cloud import bigquery

from fintrans_toolbox.src import bq_utils as bq
from fintrans_toolbox.src import table_utils as t

client = bigquery.Client()

In [ ]:
mapper = pd.read_csv('/home/jupyter/ft_digital_trade/data/raw/MCC to SIC.csv')
mapper = mapper.drop('SIC Description', axis=1)
mapper = mapper.drop('2-digit Match Confidence\n (1 exact, 0 partial match, ? requires further research/no match)', axis=1)
mapper

In [ ]:
mcc = mapper.drop('mcg', axis=1)
mcc = mcc.drop('VISA Merchant Guide Code', axis=1)
mcc = mcc.drop('Goods/ Services', axis=1)
mcc = mcc.drop('SIC Section', axis=1)

In [ ]:
spend_by_mcc = '''SELECT time_period_value, mcc, sum(spend) as total_spend
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel`
WHERE time_period = 'Quarter' 
  and mcc != 'All'
  AND merchant_channel = 'All'
  AND cardholder_origin = 'UNITED KINGDOM'
  AND cardholder_origin_country = 'All' 
GROUP BY time_period_value, mcc
ORDER BY time_period_value ASC'''
spend_by_mcc = bq.read_bq_table_sql(client, spend_by_mcc)
spend_by_mcc['Year'] = spend_by_mcc['time_period_value'].str[:4]
spend_by_mcc['Year'] =pd.to_numeric(spend_by_mcc['Year'], errors='coerce')
spend_by_mcc['mcc'] =spend_by_mcc['mcc'].str.strip()
spend_by_mcc

In [ ]:
yearly_totals = (
    spend_by_mcc.groupby(['mcc', 'Year'], as_index=False)['total_spend']
    .sum()
    .rename(columns={'total_spend': 'yearly_spend'})
)
#yearly_totals = spend_by_mcc.drop(columns=['time_period_value'], errors='ignore')
yearly_totals

In [ ]:
by_industry = pd.merge(yearly_totals, mcc, on='mcc', how='outer')
by_industry = by_industry.rename(columns={'2-digit SIC 2007': 'SIC'})
by_industry

In [ ]:
by_industry.to_csv('by_industry.csv')

In [ ]:
yearly_ind_totals = (
    by_industry.groupby(['Year', 'SIC'], as_index=False)['yearly_spend']
    .sum()
    .rename(columns={'yearly_spend': 'yearly_ind_spend'})
)
yearly_ind_totals

In [ ]:
yearly_ind_totals.to_csv('totals_by_ind.csv')

In [ ]:
ind_totals = pd.read_csv('/home/jupyter/ft_digital_trade/data/raw/16-22 ind totals.csv')
ind_totals = ind_totals.drop('2016', axis=1)
ind_totals = ind_totals.drop('2017', axis=1)
ind_totals = ind_totals.drop('2018', axis=1)
ind_totals = ind_totals.rename(columns={'2-digit SIC 2007': 'SIC'})
ind_totals

In [ ]:
ind_totals = ind_totals.transpose()
#num_inds = pd.to_numeric(ind_totals["SIC"], errors='ignore')
#num_inds
#ind_totals = ind_totals.drop('88', axis=1)
#ind_totals = ind_totals.drop('89', axis=1)
#ind_totals = ind_totals.drop('90', axis=1)
ind_totals

In [ ]:
#testing

In [ ]:
data = {
    'quarter': ['2019Q1', '2019Q2', '2019Q3', '2019Q4', '2020Q1', '2020Q2', '2020Q3', '2020Q4'],
    'year': ['2019', '2019', '2019', '2019', '2020', '2020', '2020', '2020'],
    'mcc': ['A', 'A', 'B', 'B', 'A', 'A', 'B', 'B'],
    'ind': [69, 69, 73, 73, 69, 69, 73, 73],
    'total_spend': [100, 150, 200, 250, 300, 350, 400, 450]
}
df = pd.DataFrame(data)

df['year'] =pd.to_numeric(df['year'], errors='coerce')
df['mcc'] =df['mcc'].str.strip()

yearly_totals = (
    df.groupby(['year', 'mcc'], as_index=False)['total_spend']
    .sum()
    .rename(columns={'total_spend': 'yearly_spend'})
)

yearly_ind_totals = (
    df.groupby(['year', 'ind'], as_index=False)['total_spend']
    .sum()
    .rename(columns={'total_spend': 'yearly_ind_spend'})
)

yearly_ind_totals
                             